In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler

# 1. load data
train_df = pd.read_csv("../../data/austria/raw/train_final.csv", index_col=0, parse_dates=True)
val_df = pd.read_csv("../../data/austria/raw/val_final.csv", index_col=0, parse_dates=True)
test_df = pd.read_csv("../../data/austria/raw/test_final.csv", index_col=0, parse_dates=True)

# 2. feature/target definition
target = "target_load_24h"
features = [c for c in train_df.columns if c != target]

# 3. scaling
scaler_x = StandardScaler()
scaler_y = StandardScaler()

train_sc, val_sc, test_sc = train_df.copy(), val_df.copy(), test_df.copy()

# feature scaling
train_sc[features] = scaler_x.fit_transform(train_df[features])
val_sc[features] = scaler_x.transform(val_df[features])
test_sc[features] = scaler_x.transform(test_df[features])

# target scaling
train_sc[[target]] = scaler_y.fit_transform(train_df[[target]])
val_sc[[target]] = scaler_y.transform(val_df[[target]])
test_sc[[target]] = scaler_y.transform(test_df[[target]])

# 4. Windowing function
def create_sequences(df_scaled, feature_cols, target_col, window=168):
    X, y = [], []
    data = df_scaled[feature_cols].values
    labels = df_scaled[target_col].values
    for i in range(len(df_scaled) - window):
        X.append(data[i:i+window])
        y.append(labels[i+window])
    return np.array(X), np.array(y)

# 5. create sequences
X_train, y_train = create_sequences(train_sc, features, target)
X_val, y_val     = create_sequences(val_sc, features, target)
X_test, y_test   = create_sequences(test_sc, features, target)

print(f"Sequences created:")
print(f"X_train shape: {X_train.shape}") # (Samples, 168, Features)
print(f"X_val shape:   {X_val.shape}")